# 🎬 Director-X Free Video Server — Kaggle Edition

Turn Kaggle's **free T4 GPU** (30 hrs/week!) into a video generation server for Director-X.

**Setup checklist (right sidebar):**
- ☑️ Accelerator → **GPU T4 x2**
- ☑️ Internet → **ON**
- ☑️ Have an ngrok token (free at https://ngrok.com)

Then Run All — copy the URL — paste into Director-X.

---

## Step 0: Verify GPU
If this shows "No GPU", go to sidebar → Settings → Accelerator → GPU T4 x2

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"VRAM: {vram:.1f} GB")
    print("\n✅ GPU is ready!")
else:
    print("\n❌ No GPU! Open sidebar → Settings → Accelerator → GPU T4 x2")
    print("   Also make sure Internet is ON")

## Step 1: Install packages

In [ ]:
!pip install -q flask flask-cors pyngrok diffusers transformers accelerate sentencepiece protobuf
!pip install -q imageio[ffmpeg] imageio opencv-python-headless safetensors huggingface_hub
print("\n✅ Installed")

## Step 2: ngrok token
Get yours free at https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
NGROK_AUTH_TOKEN = ""  # ← Paste your token here

if not NGROK_AUTH_TOKEN:
    print("⚠️  Paste your ngrok auth token above!")
    print("   Free at: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ ngrok ready")

## Step 3: Load model
First run downloads ~4 GB. Cached after that (~2 min reload).

In [ ]:
import torch, gc
from diffusers import LTXPipeline

dtype = torch.float16
device = "cuda"

# Clear any existing GPU memory
torch.cuda.empty_cache()
gc.collect()

print("📥 Loading LTX-Video...")
t2v_pipe = LTXPipeline.from_pretrained(
    "Lightricks/LTX-Video",
    torch_dtype=dtype,
    variant="fp16",
)
# CPU offload = only loads each layer to GPU when needed, then frees it
t2v_pipe.enable_sequential_cpu_offload()

# Enable memory efficient attention
try:
    t2v_pipe.enable_xformers_memory_efficient_attention()
    print("✅ xformers memory-efficient attention enabled")
except Exception:
    print("ℹ️  xformers not available, using default attention (still works)")

vram = torch.cuda.memory_allocated() / 1024**3
print(f"\n✅ Model loaded! VRAM: {vram:.1f} GB")

## Step 4: Quick test (optional)
Skip this if you want to go straight to the server.

In [ ]:
import time
from diffusers.utils import export_to_video

print("🎬 Test generation...")
torch.cuda.empty_cache()
gc.collect()

start = time.time()
with torch.inference_mode():
    test = t2v_pipe(
        prompt="A cinematic slow pan across a dusty frontier town at golden hour",
        negative_prompt="blurry, low quality, distorted",
        num_frames=25,
        width=512,
        height=320,
        num_inference_steps=25,
        guidance_scale=7.5,
        generator=torch.Generator(device="cuda").manual_seed(42),
    ).frames[0]

export_to_video(test, "/kaggle/working/test.mp4", fps=12)
del test
torch.cuda.empty_cache()
gc.collect()

elapsed = time.time() - start
print(f"\n✅ Done in {elapsed:.0f}s → /kaggle/working/test.mp4")

from IPython.display import HTML
from base64 import b64encode
mp4 = open("/kaggle/working/test.mp4", "rb").read()
HTML(f'<video width=512 controls><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>')

## Step 5: Start server 🚀
Copy the URL printed below → paste into Director-X → Colab (Local) provider.

In [ ]:
import os
import uuid
import json
import time
import threading
import gc
import torch
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pyngrok import ngrok
from collections import OrderedDict
from diffusers.utils import export_to_video

app = Flask(__name__)
CORS(app)

OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

jobs = OrderedDict()
MAX_JOBS = 50
gen_queue = []
gen_lock = threading.Lock()
is_generating = False

ASPECT_RATIOS = {
    "16:9": (512, 320),
    "9:16": (320, 512),
    "1:1":  (384, 384),
    "4:3":  (448, 336),
}

def get_resolution(aspect_ratio, quality="standard"):
    base = ASPECT_RATIOS.get(aspect_ratio, ASPECT_RATIOS["16:9"])
    if quality == "high":
        return (min(base[0] * 2, 768), min(base[1] * 2, 768))
    return base

def generate_video(job_id, prompt, aspect_ratio="16:9", duration=3, quality="standard", seed=-1):
    global is_generating
    try:
        is_generating = True
        jobs[job_id]["status"] = "generating"

        # Clear VRAM before generation
        torch.cuda.empty_cache()
        gc.collect()

        width, height = get_resolution(aspect_ratio, quality)
        # Clamp frames: 12fps, min 2s, max 5s on free GPU to avoid OOM
        num_frames = max(25, min(61, int(duration * 12) + 1))

        gen_kwargs = dict(
            prompt=prompt,
            negative_prompt="blurry, low quality, distorted, watermark, text overlay",
            num_frames=num_frames,
            width=width,
            height=height,
            num_inference_steps=25,
            guidance_scale=7.5,
        )
        if seed >= 0:
            gen_kwargs["generator"] = torch.Generator(device="cuda").manual_seed(seed)

        with torch.inference_mode():
            video_frames = t2v_pipe(**gen_kwargs).frames[0]

        output_path = os.path.join(OUTPUT_DIR, f"{job_id}.mp4")
        export_to_video(video_frames, output_path, fps=12)

        # Free the frames immediately
        del video_frames
        torch.cuda.empty_cache()
        gc.collect()

        jobs[job_id]["status"] = "completed"
        jobs[job_id]["video_path"] = output_path
        print(f"\u2705 Job {job_id[:8]} done: {prompt[:50]}...")

    except torch.cuda.OutOfMemoryError:
        jobs[job_id]["status"] = "failed"
        jobs[job_id]["error"] = "GPU out of memory — try shorter duration or standard quality"
        torch.cuda.empty_cache()
        gc.collect()
        print(f"\u274c Job {job_id[:8]} OOM")
    except Exception as e:
        jobs[job_id]["status"] = "failed"
        jobs[job_id]["error"] = str(e)
        print(f"\u274c Job {job_id[:8]} failed: {e}")
    finally:
        is_generating = False
        torch.cuda.empty_cache()
        gc.collect()
        process_queue()

def process_queue():
    global is_generating
    with gen_lock:
        if is_generating or not gen_queue:
            return
        job = gen_queue.pop(0)
    thread = threading.Thread(target=generate_video, kwargs=job)
    thread.start()

@app.route("/api/health", methods=["GET"])
def health():
    vram_used = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
    vram_total = torch.cuda.get_device_properties(0).total_mem / 1024**3 if torch.cuda.is_available() else 0
    return jsonify({
        "status": "ok",
        "provider": "kaggle-ltx",
        "model": "LTX-Video",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
        "vram_used_gb": round(vram_used, 1),
        "vram_total_gb": round(vram_total, 1),
        "queue_length": len(gen_queue),
        "is_generating": is_generating,
        "supported_aspects": list(ASPECT_RATIOS.keys()),
    })

@app.route("/api/video/submit", methods=["POST"])
def submit_video():
    data = request.json or {}
    prompt = data.get("prompt", "").strip()
    if not prompt:
        return jsonify({"error": "prompt is required"}), 400

    job_id = str(uuid.uuid4())
    aspect_ratio = data.get("aspect_ratio", data.get("aspectRatio", "16:9"))
    duration = min(5, max(2, int(data.get("duration", 3))))
    quality = data.get("quality", "standard")
    seed = int(data.get("seed", -1))

    jobs[job_id] = {
        "status": "queued",
        "prompt": prompt[:200],
        "aspect_ratio": aspect_ratio,
        "duration": duration,
        "created": time.time(),
        "video_path": None,
        "error": None,
    }

    while len(jobs) > MAX_JOBS:
        old_id, old_job = jobs.popitem(last=False)
        if old_job.get("video_path") and os.path.exists(old_job["video_path"]):
            os.remove(old_job["video_path"])

    gen_queue.append({
        "job_id": job_id,
        "prompt": prompt,
        "aspect_ratio": aspect_ratio,
        "duration": duration,
        "quality": quality,
        "seed": seed,
    })
    process_queue()

    return jsonify({
        "requestId": job_id,
        "statusUrl": f"/api/video/status/{job_id}",
        "provider": "kaggle-ltx",
        "model": "LTX-Video",
        "queue_position": len(gen_queue),
    })

@app.route("/api/video/status/<job_id>", methods=["GET"])
def video_status(job_id):
    if job_id not in jobs:
        return jsonify({"error": "Job not found"}), 404
    job = jobs[job_id]
    result = {
        "status": job["status"].upper(),
        "prompt": job["prompt"],
        "aspect_ratio": job.get("aspect_ratio"),
    }
    if job["status"] == "completed" and job["video_path"]:
        result["videoUrl"] = f"/api/video/download/{job_id}"
    elif job["status"] == "failed":
        result["error"] = job.get("error", "Unknown error")
    elif job["status"] == "queued":
        pos = next((i for i, j in enumerate(gen_queue) if j["job_id"] == job_id), -1)
        result["queue_position"] = pos + 1 if pos >= 0 else 0
    return jsonify(result)

@app.route("/api/video/download/<job_id>", methods=["GET"])
def download_video(job_id):
    if job_id not in jobs or not jobs[job_id].get("video_path"):
        return jsonify({"error": "Video not found"}), 404
    return send_file(jobs[job_id]["video_path"], mimetype="video/mp4")

@app.route("/api/queue", methods=["GET"])
def queue_info():
    return jsonify({
        "queue_length": len(gen_queue),
        "is_generating": is_generating,
        "recent_jobs": [
            {"id": jid, "status": j["status"], "prompt": j["prompt"][:50]}
            for jid, j in list(jobs.items())[-10:]
        ]
    })

port = 5000
public_url = ngrok.connect(port)

print("\n" + "=" * 60)
print("\U0001f3ac DIRECTOR-X VIDEO SERVER IS RUNNING!")
print("=" * 60)
print(f"\n\U0001f310 Public URL: {public_url}")
print(f"\n\U0001f4cb Paste into Director-X → Video Provider → Colab (Local):")
print(f"   {public_url}")
print(f"\n\U0001f527 Health: {public_url}/api/health")
print("\n\u26a1 Aspects: 16:9 (YT), 9:16 (TikTok/Reels), 1:1 (IG), 4:3")
print("\u26a1 Duration: 2-5 sec per clip | Quality: standard or high")
print("\n\u23f3 Keep this notebook running while generating!")
print("=" * 60)

app.run(port=port)

---
## 💡 Tips
- **30 hrs/week free GPU** — enough for 600+ clips
- **Sessions last ~12 hours.** If disconnected, re-run all cells (~2 min)
- **OOM errors?** Use `standard` quality and 2-3 sec duration
- **Phone verification** may be required first time (one-time setup)

### API
```
POST /api/video/submit  { "prompt": "...", "aspect_ratio": "16:9", "duration": 3 }
GET  /api/video/status/{id}  → { status, videoUrl }
GET  /api/video/download/{id} → mp4
GET  /api/health | /api/queue
```